In [0]:
from pyspark.sql import functions as F

In [0]:
storage_account = "dlsua5816bd"
login = "mariiasirska" 
container = login
catalog = "dbr_dev_ua5816bd"
schema = f"{login}_bronze"

In [0]:

base = f"abfss://{container}@{storage_account}.dfs.core.windows.net/lab3"
checkpoint = f"{base}/_checkpoints/eventhub_events"
table = f"`{catalog}`.`{schema}`.lab3_events_bronze"

In [0]:
scope = "scope_mariiasirska"
conn_str = dbutils.secrets.get(scope, "eventhub-connection-string")
eh_name = dbutils.secrets.get(scope, "eventhub-name")
namespace = dbutils.secrets.get(scope, "eventhub-namespace").split(".")[0]

In [0]:
jaas = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{conn_str}";'
)

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", f"{namespace}.servicebus.windows.net:9093")
    .option("subscribe", eh_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)


In [0]:
payload_schema = "event_id string, device_id string, temperature double, humidity double, event_ts string"

events = (
    raw.select(
        F.col("value").cast("string").alias("raw_body"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("enqueued_ts"),
    )
    .withColumn("e", F.from_json("raw_body", payload_schema))
    .select("e.*", "raw_body", "kafka_partition", "kafka_offset", "enqueued_ts")
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("source_system", F.lit("eventhub"))
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

In [0]:
q = (
    events.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(table)
)
q.awaitTermination()


In [0]:
print("rows:", spark.table(table).count())
display(spark.table(table).orderBy(F.desc("enqueued_ts")).limit(10))

In [0]:
for p in q.recentProgress:
    print(p["batchId"], "rows:", p["numInputRows"], "| rows/sec:", p.get("processedRowsPerSecond"))

Duplicates check:

In [0]:
display(spark.table(table).groupBy("event_id").count().filter("count > 1"))

In [0]:
display(spark.table(table).selectExpr(
    "count(*) as total",
    "count(event_id) as parsed",
    "count(distinct event_id) as distinct_events",
    "min(enqueued_ts) as first_event",
    "max(enqueued_ts) as last_event"
))

In [0]:
#for s in spark.streams.active:
#    s.stop()
#
#dbutils.fs.rm(checkpoint, True)
#spark.sql(f"TRUNCATE TABLE {table}")